# Faruq-v3 DRIV1 — DRNet Interaction Verification

Validation-only breadth screen. Coarse labels come strictly from frozen SNI `entity_family`; validation confusion pairs are never used. DRIV1 is compared primarily with DRF1. Test stays unavailable.

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)
import os, shutil, subprocess, sys, tarfile, time
from pathlib import Path
REPO=Path('/content/coffee-bean-detection'); BRANCH='agent/drnet-interaction-verification-screening'
os.chdir('/content')
if REPO.exists(): shutil.rmtree(REPO)
cmd=['git','clone','--depth','1','--branch',BRANCH,'https://github.com/ediprin/coffee-bean-detection.git',str(REPO)]
for attempt in range(3):
    if subprocess.run(cmd).returncode==0: break
    if REPO.exists(): shutil.rmtree(REPO)
    if attempt==2: raise RuntimeError('Git clone gagal')
    time.sleep(2)
subprocess.run([sys.executable,'-m','pip','install','-q','-e',str(REPO)],check=True)
sys.path.insert(0,str(REPO/'src')); os.chdir(REPO)
print('BRANCH:',BRANCH)

In [ ]:
import torch
from coffee_detector.drive_project import require_project_artifact, resolve_drive_project_root
assert torch.cuda.is_available(),'Aktifkan GPU.'
PROJECT=resolve_drive_project_root(required_relative_paths=(
 'bundles/faruq-development-v3-grouped.tar',
 'experiments/faruq-v3-yolo26n-baseline-v1/D0_seed42/weights/best.pt',
 'experiments/faruq-v3-acmc-optimization-control-v1/val_reports/acmc1_optimization_control_seed42.json',
 'experiments/faruq-v3-drnet-refinement-screening-v1/val_reports/drnet_refinement_seed42_screening.json',
))
ARCHIVE=require_project_artifact(PROJECT,'bundles/faruq-development-v3-grouped.tar')
D0=require_project_artifact(PROJECT,'experiments/faruq-v3-yolo26n-baseline-v1/D0_seed42/weights/best.pt')
CONTROL=require_project_artifact(PROJECT,'experiments/faruq-v3-acmc-optimization-control-v1/val_reports/acmc1_optimization_control_seed42.json')
DRNET=require_project_artifact(PROJECT,'experiments/faruq-v3-drnet-refinement-screening-v1/val_reports/drnet_refinement_seed42_screening.json')
DATA=Path('/content/faruq-development-v3-grouped')
if not (DATA/'faruq_grouped_summary.json').is_file():
    with tarfile.open(ARCHIVE,'r') as archive: archive.extractall('/content',filter='data')
GROUPED=DATA/'faruq_grouped_summary.json'; assert GROUPED.is_file(); assert not (DATA/'test').exists()
OUTPUT=PROJECT/'experiments/faruq-v3-drnet-interaction-verification-screening-v1'
print('GPU:',torch.cuda.get_device_name(0)); print('OUTPUT:',OUTPUT)

In [ ]:
subprocess.run([sys.executable,'-m','pytest','-q','tests/test_drnet_interaction.py'],cwd=REPO,check=True)
print('PASS: ontology mapping, fine-score restriction, native-init contract verified.')

In [ ]:
command=[sys.executable,'-u','-m','coffee_detector.experiments.run_faruq_v3_drnet_interaction_screening',
 '--data-root',str(DATA),'--grouped-summary',str(GROUPED),'--control-summary',str(CONTROL),
 '--drnet-summary',str(DRNET),'--d0-checkpoint',str(D0),'--output-root',str(OUTPUT),
 '--seed','42','--device','0','--authorize-training']
print('MENJALANKAN:',' '.join(command),flush=True)
process=subprocess.run(command,cwd=REPO,text=True,stdout=subprocess.PIPE,stderr=subprocess.STDOUT)
print(process.stdout,end='',flush=True)
if process.returncode!=0: raise RuntimeError(f'DRIV1 gagal: {process.returncode}')

In [ ]:
import json, pandas as pd
from IPython.display import display
SUMMARY=OUTPUT/'val_reports/drnet_interaction_seed42_screening.json'
result=json.loads(SUMMARY.read_text(encoding='utf-8'))
assert result['test_opened'] is False and result['test_images_accessed'] is False
rows=[{'model':name,**metrics} for name,metrics in result['controls'].items()]
rows.append({'model':'DRIV1',**result['candidate']['DRIV1']['metrics']})
display(pd.DataFrame(rows).style.format({'macro_map50_95':'{:.2%}','bottom3_class_map50_95':'{:.2%}','worst_class_map50_95':'{:.2%}'}))
print('COARSE:',result['coarse_members']); print('DECISION:',result['decision']); print('SUMMARY:',SUMMARY)